# Study Prioritized DQN

Prioritized DQN samples surprising replay items more often and corrects with weights. This compact CartPole experiment exposes the public API and the variant-specific network or replay state.

## Defining idea

$$P(i)=p_i^\alpha/\sum_k p_k^\alpha$$

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from aprenderl import PrioritizedDQN, PrioritizedDQNConfig
from aprenderl.utils import evaluate_policy

ENV_ID = "CartPole-v1"

In [ ]:
env = gym.make(ENV_ID)
config = PrioritizedDQNConfig(
    priority_alpha=0.6,
    buffer_size=10_000,
    batch_size=64,
    learning_starts=500,
    target_update_interval=250,
    exploration_steps=2_000,
    seed=7,
)
agent = PrioritizedDQN(env, config=config, device="cpu")
agent.learn(5_000, progress_bar=False)

In [ ]:
print(agent.q_network)
print("Replay items:", len(agent.replay_buffer))
print("Gradient updates:", agent.num_updates)

In [ ]:
returns = np.asarray(agent.episode_returns)
window = min(10, len(returns))
average = np.convolve(
    returns, np.ones(window) / window, mode="valid"
)
plt.figure(figsize=(8, 3))
plt.plot(returns, alpha=0.3, label="episode")
plt.plot(np.arange(window - 1, len(returns)), average, label="mean")
plt.legend()
plt.title("Prioritized DQN on CartPole")
plt.show()

In [ ]:
evaluation_env = gym.make(ENV_ID)
result = evaluate_policy(agent, evaluation_env, episodes=10)
print("Mean return:", result.mean_return)
evaluation_env.close()
env.close()

## Things to inspect

Trace one replay batch through the target and loss functions in the implementation. Change only one hyperparameter at a time and compare update stability, sample use, and evaluation return.